In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import random
import pandas as pd
from torch import nn
from sklearn.metrics import confusion_matrix
import tqdm
import torch
from pydub import AudioSegment
from google.colab import drive
import sklearn.metrics as m
import os

In [ ]:
# all file io codes are commented
# Then convert the training data into various codecs and save it to drive for future use.
# Then convert it into .npy files and save it into the drive
# we have four folders train, test, train_aug, validation and each consisting of real and fake samples.
# Then load it into session storage in colab for faster file io.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

In [ ]:
dataset_path = '/content/drive/MyDrive/for-2seconds'
os.listdir(dataset_path)


In [ ]:
audio_path = dataset_path + '/training/real/file1000.wav_16k.wav_norm.wav_mono.wav_silence.wav_2sec.wav'
SAMPLING_FREQUENCY = 16000
MEL_BANDS = 128

In [ ]:

def gen_mel(audio_path, sampling_freq, mel_bands):

    y, sr = librosa.load(audio_path,sr=sampling_freq)

    target_len = 2 * sampling_freq

    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))

    else:
        y = y[:target_len]

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=512,
        hop_length=160,
        win_length=400,
        n_mels=mel_bands,
        fmin=20,
        fmax=8000
    )

    return mel
def gen_meldb(mel):
  mel_db = librosa.power_to_db(mel, ref=np.max)
  return mel_db


In [ ]:
def plot_mel_spectrum(mel, sr):
  mel_db = librosa.power_to_db(mel, ref=np.max)
  plt.figure(figsize=(10,4))
  librosa.display.specshow(
      mel_db,
      sr=sr,
      x_axis='time',
      y_axis='mel'
  )
  plt.colorbar(format='%+2.0f dB')
  plt.title('Mel Spectrogram')
  plt.show()
def plot_mel_spectrum_v2(mel_db, sr):
  plt.figure(figsize=(10,4))
  librosa.display.specshow(
      mel_db,
      sr=sr,
      x_axis='time',
      y_axis='mel'
  )
  plt.colorbar(format='%+2.0f dB')
  plt.title('Mel Spectrogram')
  plt.show()

In [ ]:
audio_files_training_real = list(
    Path("/content/drive/MyDrive/for-2seconds/training/real")
    .rglob("*.wav")
)
audio_files_training_fake = list(
    Path("/content/drive/MyDrive/for-2seconds/training/fake")
    .rglob("*.wav")
)
audio_files_testing_real = list(
    Path("/content/drive/MyDrive/for-2seconds/testing/real")
    .rglob("*.wav")
)
audio_files_testing_fake = list(
    Path("/content/drive/MyDrive/for-2seconds/testing/fake")
    .rglob("*.wav")
)
audio_files_validation_real = list(
    Path("/content/drive/MyDrive/for-2seconds/validation/real")
    .rglob("*.wav")
)
audio_files_validation_fake = list(
    Path("/content/drive/MyDrive/for-2seconds/validation/fake")
    .rglob("*.wav")
)


In [ ]:
from pathlib import Path
audio_files_train_aug_real = list(
    Path("/content/drive/MyDrive/for-2seconds/training_aug/real").
    glob("*")
)
audio_files_train_aug_real = [
    f for f in audio_files_train_aug_real
    if str(f) != "/content/drive/MyDrive/for-2seconds/training_aug/real/training_real_6193.mp3"
]
audio_files_train_aug_fake = list(
    Path("/content/drive/MyDrive/for-2seconds/training_aug/fake").
    glob("*")
)


In [ ]:
NUM_FILES = 20
for i in range(NUM_FILES):
  random_num = random.randint(0 , len(audio_files_training_fake) - 1)
  mel = gen_mel(audio_files_training_fake[random_num], SAMPLING_FREQUENCY, MEL_BANDS)
  print("training")
  random_num = random.randint(0 , len(audio_files_testing_fake) - 1)
  plot_mel_spectrum(mel, SAMPLING_FREQUENCY)
  mel = gen_mel(audio_files_testing_fake[random_num], SAMPLING_FREQUENCY, MEL_BANDS)
  print("testing")
  plot_mel_spectrum(mel, SAMPLING_FREQUENCY)
  print("============================================================")


In [ ]:
!apt-get install ffmpeg
!pip install pydub

In [ ]:
def convert_audio(input_file, output_file, format_name):
    audio = AudioSegment.from_file(input_file)
    audio.export(output_file, format=format_name)

In [ ]:

Path("/content/training_aug/fake").mkdir(
    parents=True,
    exist_ok=True
)
Path("/content/training_aug/real").mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:

for i, file in (enumerate(tqdm.tqdm(audio_files_training_real))):

  a = random.random()
  if a < 0.25:
    convert_audio(file, f"/content/training_aug/real/training_real_{i}.mp3", "mp3")
  elif a < 0.5:
    convert_audio(file, f"/content/training_aug/real/training_real_{i}.ogg", "ogg")
  elif a < 0.75:
    convert_audio(file, f"/content/training_aug/real/training_real_{i}.m4a", "ipod")
  else:
    convert_audio(file, f"/content/training_aug/real/training_real_{i}.wav", "wav")

for i, file in (enumerate(tqdm.tqdm(audio_files_training_fake))):
  a = random.random()
  if a < 0.25:
    convert_audio(file, f"/content/training_aug/fake/training_fake_{i}.mp3", "mp3")
  elif a < 0.5:
    convert_audio(file, f"/content/training_aug/fake/training_fake_{i}.ogg", "ogg")
  elif a < 0.75:
    convert_audio(file, f"/content/training_aug/fake/training_fake_{i}.m4a", "ipod")
  else:
    convert_audio(file, f"/content/training_aug/fake/training_fake_{i}.wav", "wav")



In [ ]:
# !cp "/content/training_aug/" "/content/drive/MyDrive/for-2seconds/training_aug/"

In [ ]:
file_path = '/content'
drive_path_save = '/content/drive/MyDrive'
save_path_train_real = '/processed/train/real'
save_path_train_fake = '/processed/train/fake'
save_path_test_real = '/processed/test/real'
save_path_test_fake = '/processed/test/fake'
save_path_validation_real = '/processed/validation/real'
save_path_validation_fake = '/processed/validation/fake'



def save_the_npy(save_path, file_list, prefix):
  Path(save_path).mkdir(
        parents=True,
        exist_ok=True
    )
  for i,file_name in (enumerate(tqdm.tqdm(file_list))):
    mel = gen_mel(file_name, SAMPLING_FREQUENCY, MEL_BANDS)
    mel_db = gen_meldb(mel)
    save_name = save_path +'/' +prefix + str(i + 1) + ".npy"
    np.save(save_name, mel_db.astype(np.float32))






In [ ]:
# save_the_npy(drive_path_save + save_path_test_real, audio_files_testing_real, "testing_real")
# save_the_npy(drive_path_save + save_path_test_fake, audio_files_testing_fake, "testing_fake")
# save_the_npy(drive_path_save + save_path_train_real, audio_files_training_real, "testing_real")
# save_the_npy(drive_path_save + save_path_train_fake, audio_files_training_fake, "testing_fake")
# save_the_npy(drive_path_save + save_path_validation_real, audio_files_validation_real, "validation_real")
# save_the_npy(drive_path_save + save_path_validation_fake, audio_files_validation_fake, "validation_fake")


In [ ]:

# save_aug_real_path = "/processed/train_aug/real"
# save_aug_fake_path = "/processed/train_aug/fake"
# save_the_npy(drive_path_save + save_aug_real_path, audio_files_train_aug_real, "training_aug_real")
# save_the_npy(drive_path_save + save_aug_fake_path, audio_files_train_aug_fake, "training_aug_fake")

In [ ]:

save_aug_real_path = "/processed/train_aug/real"
save_aug_fake_path = "/processed/train_aug/fake"

In [ ]:
# !cp -r "/content/drive/MyDrive/Colab Notebooks/processed/" "/content/"

In [ ]:

data_train = []
for file_name in Path(file_path + save_path_train_real + '/').glob("*.npy"):
  data_train.append({
      "path" : str(file_name),
      "label" : 0
  })
for file_name in Path(file_path + save_path_train_fake + '/').glob("*.npy"):
  data_train.append({
      "path" : str(file_name),
      "label" : 1
  })

df_train = pd.DataFrame(data_train)



In [ ]:
data_test = []
for file_name in Path(file_path + save_path_test_real + '/').glob("*.npy"):
  data_test.append({
      "path" : str(file_name),
      "label" : 0
  })
for file_name in Path(file_path + save_path_test_fake + '/').glob("*.npy"):
  data_test.append({
      "path" : str(file_name),
      "label" : 1
  })

df_test = pd.DataFrame(data_test)



In [ ]:
data_validation = []
for file_name in Path(file_path + save_path_validation_real + '/').glob("*.npy"):
  data_validation.append({
      "path" : str(file_name),
      "label" : 0
  })
for file_name in Path(file_path + save_path_validation_fake + '/').glob("*.npy"):
  data_validation.append({
      "path" : str(file_name),
      "label" : 1
  })

df_validation = pd.DataFrame(data_validation)
print(df_validation)
print(df_validation.shape)


In [ ]:
data_train_aug = []
drive_path = "/content"
for file_name in Path(file_path + save_aug_real_path + '/').glob("*.npy"):
  data_train_aug.append({
      "path" : str(file_name),
      "label" : 0
  })
for file_name_aug in Path(file_path + save_aug_fake_path + '/').glob("*.npy"):
  data_train_aug.append({
      "path" : str(file_name_aug),
      "label" : 1
  })

df_train_aug = pd.DataFrame(data_train_aug)
print(df_train_aug)
print(df_train_aug.shape)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:

feature_map = {
    0 : "real",
    1 : "fake"
}

In [ ]:
NUM_FILES = 20
for i in range(NUM_FILES):
  random_num = random.randint(0 , 13955)
  file_name = df_train["path"][random_num]
  mel_db = np.load(file_name)
  plot_mel_spectrum_v2(mel_db, SAMPLING_FREQUENCY)
  print(feature_map[df_train["label"][random_num]])
  print("====================================================")

In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset

class DeepfakeDataset(Dataset):

    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        path = self.df.iloc[idx]["path"]
        label = self.df.iloc[idx]["label"]
        mel = np.load(path)
        mel = (mel - mel.mean()) / (mel.std() + 1e-8)
        mel = torch.tensor(mel,dtype=torch.float32)
        mel = mel.unsqueeze(0)
        label = torch.tensor(label,dtype=torch.long)

        return mel, label

In [ ]:
train_dataset = DeepfakeDataset(df_train)
test_dataset = DeepfakeDataset(df_test)
validation_dataset = DeepfakeDataset(df_validation)
train_aug_dataset = DeepfakeDataset(df_train_aug)


In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_dataloader = DataLoader(train_dataset,
      batch_size = BATCH_SIZE,
      shuffle = True,
      num_workers = 2,
      pin_memory = True)

test_dataloader = DataLoader(test_dataset,
      batch_size = BATCH_SIZE,
      shuffle = False,
      num_workers = 2,
      pin_memory = True)

validation_dataloader = DataLoader(validation_dataset,
      batch_size = BATCH_SIZE,
      shuffle = False,
      num_workers = 2,
      pin_memory = True)
train_aug_dataloader = DataLoader(train_aug_dataset,
      batch_size = BATCH_SIZE,
      shuffle = True,
      num_workers = 2,
      pin_memory = True)

train_dataloader

In [ ]:
class ClassifierModelV1(nn.Module):
  def __init__(self, input_shape : int, output_shape : int):
    super().__init__()
    self.block_1 = nn.Sequential(
    nn.Conv2d(input_shape, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(64, 128, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    )
    self.classifier = nn.Sequential(
    nn.Linear(51200, 256),
    nn.ReLU(),
    nn.Dropout(),
    nn.Linear(256, output_shape)
    )


  def forward(self, x : torch.tensor) -> torch.tensor:
    x = self.block_1(x)
    x = torch.flatten(x, start_dim=1)
    x = self.classifier(x)
    return x

model_1 = ClassifierModelV1(input_shape = 1,
                           output_shape = 1).to(device)

model_1

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()

In [ ]:
def evaluation(model, test_dataloader):

  all_preds = []
  all_labels = []
  all_scores = []

  model.eval()

  with torch.inference_mode():
      for X, y in test_dataloader:

          X = X.to(device)

          logits = model(X)
          scores = torch.sigmoid(logits).cpu().numpy()
          all_scores.extend(scores.flatten())
          preds = (torch.sigmoid(logits.squeeze(1)) > 0.5).long().cpu()
          all_preds.extend(preds.numpy())
          all_labels.extend(y.numpy())


  fp, tp, thresholds = m.roc_curve(all_labels,all_scores)
  fn = 1 - tp
  eer_idx = np.nanargmin(np.abs(fn - fp))
  eer = (fn[eer_idx] + fp[eer_idx]) / 2
  cm = m.confusion_matrix(all_labels, all_preds, labels=[0,1])
  real_acc = cm[0,0] / max(cm[0].sum(), 1)
  fake_acc = cm[1,1] / max(cm[1].sum(), 1)
  accuracy = m.accuracy_score(all_labels, all_preds)


  f1_score = m.f1_score(all_labels, all_preds)




  print(f"Confusion Matrix : {cm}")
  print(f"Real accuracy : {real_acc:.4f}")
  print(f"Fake accuracy : {fake_acc:.4f}")
  print(f"F1 score : {f1_score}")
  print(f"EER: {eer}")
  print(f"Overall Accuracy : {accuracy}")

In [ ]:
def train_step(model, data_loader, optimizer, loss_fn):
    train_loss = 0;
    for batch, (X, y) in (enumerate(tqdm.tqdm(data_loader))):
      X = X.to(device)
      y = y.to(device)
      model.train();
      y_pred = model(X)
      loss = loss_fn(y_pred.squeeze(1), y.float())
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      train_loss +=loss.item()
    train_loss /= len(data_loader)
    print(f"training_loss: {train_loss}")

def test_step(model, data_loader, loss_fn):
  test_loss = 0
  model_accuracy = 0
  model.eval()
  with torch.inference_mode():
    for batch, (X, y) in (enumerate(tqdm.tqdm(data_loader))):
      X = X.to(device)
      y = y.to(device)
      y_pred = model(X)
      preds = (torch.sigmoid(y_pred.squeeze(1)) > 0.5).long()
      test_loss += loss_fn(y_pred.squeeze(1), y.float()).item()
      model_accuracy += m.accuracy_score(preds.cpu(), y.cpu().numpy())
  test_loss /= len(data_loader)
  model_accuracy /= len(data_loader)
  print(f"test loss : {test_loss} | model accuracy : {model_accuracy}")


In [ ]:
model_2 = ClassifierModelV1(input_shape = 1,
                           output_shape = 1).to(device)

In [ ]:

optimizer_2 = torch.optim.Adam(params = model_2.parameters(), lr = 0.001)

In [ ]:
EPOCHS = 3
torch.manual_seed(42)
for epoch in range(EPOCHS):
  train_step(
      model = model_2,
      data_loader = train_aug_dataloader,
      optimizer = optimizer_2,
      loss_fn = loss_fn
  )
  test_step(
      model = model_2,
      data_loader = test_dataloader,
      loss_fn = loss_fn
  )

In [ ]:
evaluation(model_2, test_dataloader)